In [ ]:
from pathlib import Path
import json, shutil, sys, time
import numpy as np
import pandas as pd

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
OUT = WORK / 'day12-conformal-fdr-benchmark'
OUT.mkdir(parents=True, exist_ok=True)

# Locate the attached Day 10 feature checkpoint and its scored-model metadata.
data_candidates = []
for fp in INPUT.rglob('features.csv'):
    root = fp.parent
    required = [root / 'manifest.csv', root / 'series.npy']
    if not all(p.is_file() for p in required):
        continue
    metadata_path = root / 'scores' / 'run.json'
    try:
        ids = pd.read_csv(fp, usecols=['source_id'], dtype={'source_id': 'string'})['source_id'].astype('string')
        n_rows = len(ids)
        n_sources = int(ids.str.replace(r'\.0$', '', regex=True).str.fullmatch(r'\d{8,16}', na=False).sum())
        data_candidates.append((n_rows, n_sources, root, metadata_path if metadata_path.is_file() else None))
    except Exception as exc:
        print('Skipping unreadable data candidate:', root, type(exc).__name__, str(exc)[:160])
if not data_candidates:
    raise FileNotFoundError('Attach the successful Day 10 ML Evidence output containing manifest.csv, features.csv, and series.npy.')
print('Attached data candidates (rows, numeric TICs, path):', [(n, t, str(root)) for n, t, root, _ in sorted(data_candidates, key=lambda x: (x[0], x[1], str(x[2])), reverse=True)])
rows, _, DATA, RUN_META_PATH = max(data_candidates, key=lambda x: (x[0], x[1], str(x[2])))

project_candidates = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/cosmic_black_swan/injections.py').is_file()]
if not project_candidates:
    raise FileNotFoundError('Attach Cosmic Swan 4 source with pyproject.toml and src/cosmic_black_swan/injections.py.')
PROJECT_SOURCE = max(project_candidates, key=lambda p: len(list((p / 'src/cosmic_black_swan').glob('*.py'))))
sys.path.insert(0, str(PROJECT_SOURCE / 'src'))

features = pd.read_csv(DATA / 'features.csv', dtype={'source_id': 'string'})
series = np.load(DATA / 'series.npy', mmap_mode='r')
run_meta = json.loads(RUN_META_PATH.read_text(encoding='utf-8')) if RUN_META_PATH else {}
print('Scoring metadata:', RUN_META_PATH if RUN_META_PATH else 'not present; using project-standard numeric feature selection')
if len(features) != len(series):
    raise ValueError(f'features/series are misaligned: {len(features)} vs {len(series)}')

features['_row_position'] = np.arange(len(features), dtype=int)
features['source_id'] = features['source_id'].astype('string').str.replace(r'\.0$', '', regex=True)
features = features[features['source_id'].str.fullmatch(r'\d{8,16}', na=False)].copy()
# Keep one deterministic sector row per TIC so no source crosses the split.
sort_cols = [c for c in ('source_id', 'sector', 'observation_id') if c in features.columns]
features = features.sort_values(sort_cols).drop_duplicates('source_id', keep='first').reset_index(drop=True)

from cosmic_black_swan.injections import _inject, _update_features
from cosmic_black_swan.conformal import conformal_upper_tail_pvalues
from cosmic_black_swan.score import NON_FEATURE_COLUMNS, QUALITY_FEATURE_COLUMNS

configured = run_meta.get('numeric_feature_columns', [])
excluded = set(NON_FEATURE_COLUMNS) | set(QUALITY_FEATURE_COLUMNS) | {
    'sector', 'camera', 'ccd', 'quality_fallback', 'segment_count', 'n_cadences_raw', 'n_cadences_valid', '_row_position'
}
if configured:
    FEATURE_COLUMNS = [c for c in configured if c in features.columns and c not in excluded]
else:
    FEATURE_COLUMNS = [c for c in features.select_dtypes(include=[np.number]).columns if c not in excluded]
if len(features) < 1000 or len(FEATURE_COLUMNS) < 5:
    raise ValueError(f'Insufficient source-level data/features: sources={len(features)}, features={len(FEATURE_COLUMNS)}')
if 'relative_robust_sigma' not in features:
    raise ValueError('Injection feature updates require relative_robust_sigma in features.csv')
print('Data:', DATA)
print('Project:', PROJECT_SOURCE)
print(f'Observations={rows:,}; unique TICs after one-row-per-source split={len(features):,}; detector features={len(FEATURE_COLUMNS)}')
print('Definition of a benchmark null: a held-out TESS background with no added synthetic injection.')

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

SPLIT_SEED = 20261115
rng = np.random.default_rng(SPLIT_SEED)
order = rng.permutation(len(features))
n_train = int(0.60 * len(order))
n_cal = int(0.20 * len(order))
train = features.iloc[order[:n_train]].copy()
calibration = features.iloc[order[n_train:n_train+n_cal]].copy()
test = features.iloc[order[n_train+n_cal:]].copy()

model = make_pipeline(
    SimpleImputer(strategy='median'),
    RobustScaler(quantile_range=(10.0, 90.0)),
    IsolationForest(n_estimators=300, contamination='auto', random_state=SPLIT_SEED, n_jobs=-1),
)
X_train = train[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
X_cal = calibration[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
X_test = test[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
model.fit(X_train)
calibration_scores = -model.decision_function(X_cal)
test_control_scores = -model.decision_function(X_test)
if not np.isfinite(calibration_scores).all() or not np.isfinite(test_control_scores).all():
    raise ValueError('Non-finite anomaly scores found; stop before making an FDR report.')

print(f'[PASS] Source-disjoint split: train={len(train):,}, calibration={len(calibration):,}, held-out backgrounds={len(test):,}')
print(f'[PASS] Minimum conformal p-value: {1/(len(calibration_scores)+1):.6g}')
print('[INFO] Model is frozen before calibration and test scoring.')

In [ ]:
def benjamini_yekutieli(pvalues, q):
    p = np.asarray(pvalues, dtype=float)
    if p.ndim != 1 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('p-values must be finite, one-dimensional, and in [0,1]')
    if not 0 < q < 1:
        raise ValueError('q must be between 0 and 1')
    m = len(p)
    if m == 0:
        return np.zeros(0, dtype=bool)
    harmonic = np.sum(1.0 / np.arange(1, m + 1))
    order = np.argsort(p)
    thresholds = q * np.arange(1, m + 1) / (m * harmonic)
    passed = np.flatnonzero(p[order] <= thresholds)
    rejected = np.zeros(m, dtype=bool)
    if passed.size:
        rejected[order[:passed[-1] + 1]] = True
    return rejected

CATALOG_SIZE = min(500, len(test))
N_REPLICATES = 100
INJECTED_FRACTION = 0.10
Q_LEVELS = (0.05, 0.10)
FAMILIES = ('eclipse', 'flare', 'periodic', 'step_artifact')
AMPLITUDES_SIGMA = (1.0, 2.0, 4.0, 8.0)
if CATALOG_SIZE < 200:
    raise ValueError(f'Only {CATALOG_SIZE} held-out backgrounds; need at least 200 for the benchmark.')

rep_rng = np.random.default_rng(20261116)
run_rows, prediction_rows = [], []
test = test.reset_index(drop=True)
test_control_scores = np.asarray(test_control_scores)
for replicate in range(N_REPLICATES):
    chosen = rep_rng.choice(len(test), size=CATALOG_SIZE, replace=False)
    n_alt = max(1, int(round(CATALOG_SIZE * INJECTED_FRACTION)))
    alt_positions = set(rep_rng.choice(CATALOG_SIZE, size=n_alt, replace=False).tolist())
    catalog_scores = test_control_scores[chosen].copy()
    truth_injected = np.zeros(CATALOG_SIZE, dtype=bool)
    chosen_families, chosen_amplitudes = ['none'] * CATALOG_SIZE, [0.0] * CATALOG_SIZE
    injected_positions, injected_feature_rows = [], []

    for catalog_pos in alt_positions:
        row = test.iloc[int(chosen[catalog_pos])]
        seq = np.asarray(series[int(row['_row_position'])], dtype=np.float32)
        family = str(rep_rng.choice(FAMILIES))
        amplitude = float(rep_rng.choice(AMPLITUDES_SIGMA))
        injected = _inject(seq, family, amplitude, rep_rng)
        values = row.to_dict()
        values.update(_update_features(row, injected))
        injected_positions.append(catalog_pos)
        injected_feature_rows.append({c: values.get(c, np.nan) for c in FEATURE_COLUMNS})
        truth_injected[catalog_pos] = True
        chosen_families[catalog_pos], chosen_amplitudes[catalog_pos] = family, amplitude

    injected_matrix = pd.DataFrame(injected_feature_rows, columns=FEATURE_COLUMNS)
    injected_matrix = injected_matrix.replace([np.inf, -np.inf], np.nan)
    catalog_scores[injected_positions] = -model.decision_function(injected_matrix)
    pvalues = conformal_upper_tail_pvalues(calibration_scores, catalog_scores)
    for q in Q_LEVELS:
        rejected = benjamini_yekutieli(pvalues, q)
        n_rejected = int(rejected.sum())
        n_false = int(np.sum(rejected & ~truth_injected))
        n_true = int(np.sum(rejected & truth_injected))
        n_alt_actual = int(truth_injected.sum())
        fdp = n_false / max(n_rejected, 1)
        power = n_true / max(n_alt_actual, 1)
        run_rows.append({
            'replicate': replicate, 'q': q, 'catalog_size': CATALOG_SIZE,
            'null_hypotheses': CATALOG_SIZE - n_alt_actual, 'injected_hypotheses': n_alt_actual,
            'discoveries': n_rejected, 'false_discoveries': n_false, 'true_injected_detections': n_true,
            'fdp': fdp, 'power': power,
        })
        for pos in range(CATALOG_SIZE):
            prediction_rows.append({
                'replicate': replicate, 'q': q, 'source_id': str(test.iloc[int(chosen[pos])]['source_id']),
                'is_injected': bool(truth_injected[pos]), 'injection_family': chosen_families[pos],
                'amplitude_sigma': chosen_amplitudes[pos], 'anomaly_score': float(catalog_scores[pos]),
                'conformal_pvalue': float(pvalues[pos]), 'rejected_by_BY': bool(rejected[pos]),
            })
    if (replicate + 1) % 10 == 0:
        print(f'Completed injection-control catalogs: {replicate + 1}/{N_REPLICATES}', flush=True)

catalog_results = pd.DataFrame(run_rows)
predictions = pd.DataFrame(prediction_rows)
catalog_results.to_csv(OUT / 'day12_fdr_catalog_metrics.csv', index=False)
predictions.to_csv(OUT / 'day12_fdr_test_predictions.csv', index=False)
print('[PASS] Completed the controlled injection FDR benchmark.')

In [ ]:
summary_rows = []
for q, group in catalog_results.groupby('q', sort=True):
    fdp_values = group['fdp'].to_numpy(dtype=float)
    power_values = group['power'].to_numpy(dtype=float)
    boot_rng = np.random.default_rng(771 + int(round(q * 1000)))
    boot = boot_rng.choice(fdp_values, size=(3000, len(fdp_values)), replace=True).mean(axis=1)
    lower, upper = np.quantile(boot, [0.025, 0.975])
    summary_rows.append({
        'q_target': float(q),
        'empirical_mean_FDR': float(fdp_values.mean()),
        'FDR_monte_carlo_95pct_interval_low': float(lower),
        'FDR_monte_carlo_95pct_interval_high': float(upper),
        'mean_power_for_injected_signals': float(power_values.mean()),
        'mean_discoveries_per_catalog': float(group['discoveries'].mean()),
        'fraction_catalogs_with_any_discovery': float((group['discoveries'] > 0).mean()),
        'point_estimate_at_or_below_q': bool(fdp_values.mean() <= q),
        'empirical_screen_pass_95pct_upper_bound': bool(upper <= q),
    })
summary_df = pd.DataFrame(summary_rows)
summary = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'source_data': str(DATA),
    'project_source': str(PROJECT_SOURCE),
    'split_seed': SPLIT_SEED,
    'train_sources': int(len(train)), 'calibration_sources': int(len(calibration)), 'heldout_background_sources': int(len(test)),
    'catalog_size': CATALOG_SIZE, 'replicates': N_REPLICATES,
    'injected_fraction': INJECTED_FRACTION, 'injection_families': list(FAMILIES), 'amplitudes_sigma': list(AMPLITUDES_SIGMA),
    'calibration_method': 'split-conformal upper-tail p-values from non-injected calibration backgrounds',
    'multiple_testing_method': 'Benjamini-Yekutieli, valid under arbitrary dependence when each null p-value is super-uniform',
    'null_definition': 'a held-out TESS background with no added synthetic injection',
    'interpretation_limit': 'This estimates FDR for detecting added synthetic templates on sampled TESS backgrounds. Backgrounds can contain natural variability, so this is not a measured astrophysical false discovery rate for the full real TESS catalog.',
    'summary': summary_df.to_dict(orient='records'),
    'outputs': ['day12_fdr_summary.json', 'day12_fdr_summary.csv', 'day12_fdr_catalog_metrics.csv', 'day12_fdr_test_predictions.csv'],
}
(OUT / 'day12_fdr_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary_df.to_csv(OUT / 'day12_fdr_summary.csv', index=False)
report = [
    '# Day 12 conformal FDR benchmark', '',
    f"- Source-disjoint groups: {len(train):,} training, {len(calibration):,} calibration, {len(test):,} held-out background TICs.",
    f'- Controlled catalogs: {N_REPLICATES} replicates × {CATALOG_SIZE} hypotheses; injected fraction {INJECTED_FRACTION:.0%}.',
    '- Calibration and null tests use non-injected TESS-derived backgrounds; positives are synthetic injections.',
    '- BY is used to account for arbitrary dependence among p-values, assuming the null p-values are super-uniform.',
    '- This benchmark does not measure false discovery rate for astrophysical discoveries in the full real TESS catalog.', '',
    '| q target | mean FDP | Monte Carlo 95% interval | injected-signal power | mean discoveries | empirical screen |',
    '|---:|---:|:---:|---:|---:|:---:|',
    *[f"| {r['q_target']:.2f} | {r['empirical_mean_FDR']:.4f} | [{r['FDR_monte_carlo_95pct_interval_low']:.4f}, {r['FDR_monte_carlo_95pct_interval_high']:.4f}] | {r['mean_power_for_injected_signals']:.4f} | {r['mean_discoveries_per_catalog']:.2f} | {'PASS' if r['empirical_screen_pass_95pct_upper_bound'] else 'REVIEW'} |" for r in summary_rows], '',
    'Interpretation: compare the average FDP across repeated catalogs with each nominal q. A single catalog can exceed q; FDR is an expectation over repeated catalogs. Review the Monte Carlo interval and injected-signal power before claiming success.',
]
(OUT / 'day12_fdr_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('Saved benchmark outputs under:', OUT)
print(summary_df.to_string(index=False))
print('Scope:', summary['interpretation_limit'])

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

path = "/kaggle/working/day12-conformal-fdr-benchmark/day12_fdr_test_predictions.csv"
df = pd.read_csv(path)
df = df[df["q"].round(2) == 0.10]

null_scores = df.loc[~df["is_injected"], "anomaly_score"].to_numpy()
rows = []

for (family, amplitude), injected in df[df["is_injected"]].groupby(
    ["injection_family", "amplitude_sigma"]
):
    labels = [0] * len(null_scores) + [1] * len(injected)
    scores = list(null_scores) + list(injected["anomaly_score"])
    rows.append({
        "family": family,
        "amplitude_sigma": amplitude,
        "AUC_vs_background": roc_auc_score(labels, scores),
        "BY_detection_rate": injected["rejected_by_BY"].mean(),
        "median_conformal_p": injected["conformal_pvalue"].median(),
    })

print(pd.DataFrame(rows).sort_values(
    ["family", "amplitude_sigma"]
).to_string(index=False))